In [1]:
import json
import random
import torch
import numpy as np
import nltk
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel
from huggingface_hub import login
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import os

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

SEEDS = [42, 123, 456]


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class ModelEvaluator:
    def __init__(self, base_model_name: str, trained_model_path: str = None):
        try:
            HF_TOKEN = "HF_TOKEN"
            login(token=HF_TOKEN)
        except:
            pass

        print(f"Loading model: {base_model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True
        )

        if trained_model_path:
            print(f"Loading LoRA adapter from: {trained_model_path}")
            self.model = PeftModel.from_pretrained(
                base_model,
                trained_model_path,
                is_trainable=False
            )
        else:
            self.model = base_model

        print("Model loaded\n")

    def generate_text(self, prompt: str, max_new_tokens: int = 100) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.8,
                do_sample=True,
                top_p=0.9,
                pad_token_id=self.tokenizer.eos_token_id
            )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def generate_from_prompts(self, prompts: list) -> list:
        print(f"  Generating {len(prompts)} outputs...")
        generated = []
        for i, prompt in enumerate(prompts):
            text = self.generate_text(prompt)
            generated.append(text)
            if (i + 1) % 10 == 0:
                print(f"    {i + 1}/{len(prompts)} done")
        return generated

    def calculate_lexical_diversity(self, texts: list) -> dict:
        all_tokens = []
        for text in texts:
            tokens = text.lower().split()
            all_tokens.extend(tokens)

        types = len(set(all_tokens))
        tokens_count = len(all_tokens)
        ttr = types / tokens_count if tokens_count > 0 else 0

        bigrams = []
        for text in texts:
            words = text.lower().split()
            bigrams.extend([f"{words[i]}_{words[i+1]}" for i in range(len(words)-1)])
        unique_bigrams = len(set(bigrams))
        total_bigrams = len(bigrams)

        def distinct_n(n):
            all_ngrams = []
            for text in texts:
                words = text.lower().split()
                all_ngrams.extend(zip(*[words[i:] for i in range(n)]))
            return round(len(set(all_ngrams)) / len(all_ngrams), 4) if all_ngrams else 0

        tokenized = [t.lower().split() for t in texts]
        smoother = SmoothingFunction().method1
        self_bleu_scores = []
        sample = tokenized[:min(50, len(tokenized))]
        for i, hyp in enumerate(sample):
            refs = sample[:i] + sample[i+1:]
            if refs:
                self_bleu_scores.append(sentence_bleu(refs, hyp, smoothing_function=smoother))
        self_bleu_score = round(float(np.mean(self_bleu_scores)), 4) if self_bleu_scores else 0

        return {
            'type_token_ratio': round(ttr, 4),
            'vocabulary_size': types,
            'total_tokens': tokens_count,
            'bigram_diversity': round(unique_bigrams / total_bigrams if total_bigrams > 0 else 0, 4),
            'distinct_1': distinct_n(1),
            'distinct_2': distinct_n(2),
            'distinct_3': distinct_n(3),
            'self_bleu': self_bleu_score
        }

    def calculate_perplexity(self, texts: list) -> float:
        total_loss = 0
        count = 0
        for text in texts:
            inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = self.model(**inputs, labels=inputs["input_ids"])
                total_loss += outputs.loss.item()
                count += 1
        return round(float(np.exp(total_loss / count)), 2)


# ── NLI factual consistency ──────────────────────────────────────────────────

def load_nli_model(device: int = 0):
    """Load DeBERTa NLI model ONCE and reuse across all evaluations (saves ~11 redundant loads)."""
    print("Loading NLI model (cross-encoder/nli-deberta-v3-small)...")
    nli = pipeline(
        "text-classification",
        model="cross-encoder/nli-deberta-v3-small",
        device=device,
        top_k=None
    )
    print("NLI model ready.\n")
    return nli


def calculate_factual_consistency(generated_texts: list, reference_texts: list, nli) -> float:
    """NLI-based factual consistency: mean entailment score of each generated sentence vs. its prompt."""
    scores = []
    for gen, ref in zip(generated_texts, reference_texts):
        sentences = nltk.sent_tokenize(gen)
        for sent in sentences:
            if len(sent.strip()) < 10:
                continue
            pair = f"{ref} [SEP] {sent}"
            result = nli(pair[:512], truncation=True)
            if isinstance(result[0], list):   # top_k=None wraps in an extra list
                result = result[0]
            for r in result:
                if r['label'].upper() == 'ENTAILMENT':
                    scores.append(r['score'])
                    break
    return round(float(np.mean(scores)), 4) if scores else 0.0


# ── Prompt loading ───────────────────────────────────────────────────────────

def load_test_prompts(filepath: str, n: int = 50) -> list:
    with open(filepath, 'r') as f:
        data = [json.loads(line) for line in f]
    prompts = []
    for item in data[:n * 2]:
        words = item['text'].split()
        if len(words) >= 20:
            prompts.append(' '.join(words[:15]))
            if len(prompts) >= n:
                break
    return prompts


# ── Single-seed run ──────────────────────────────────────────────────────────

def run_single_seed_evaluation(evaluator: ModelEvaluator, prompts: list, nli) -> dict:
    generated = evaluator.generate_from_prompts(prompts)
    lexical   = evaluator.calculate_lexical_diversity(generated)
    perplexity = evaluator.calculate_perplexity(generated)
    factual   = calculate_factual_consistency(generated, prompts, nli)
    return {
        'lexical_diversity':   lexical,
        'perplexity':          perplexity,
        'factual_consistency': factual,
        'num_prompts':         len(prompts),
        'generated_texts':     generated
    }


def aggregate_seed_results(seed_results: list) -> dict:
    """Compute mean ± std across seed runs for every metric."""
    metric_keys = ['type_token_ratio', 'vocabulary_size', 'bigram_diversity',
                   'distinct_1', 'distinct_2', 'distinct_3', 'self_bleu']
    agg = {'lexical_diversity': {}, 'perplexity': {}, 'factual_consistency': {}}

    for key in metric_keys:
        vals = [r['lexical_diversity'][key] for r in seed_results]
        agg['lexical_diversity'][key] = {
            'mean': round(float(np.mean(vals)), 4),
            'std':  round(float(np.std(vals)),  4)
        }

    ppl_vals = [r['perplexity'] for r in seed_results]
    agg['perplexity'] = {
        'mean': round(float(np.mean(ppl_vals)), 2),
        'std':  round(float(np.std(ppl_vals)),  2)
    }

    fc_vals = [r['factual_consistency'] for r in seed_results]
    agg['factual_consistency'] = {
        'mean': round(float(np.mean(fc_vals)), 4),
        'std':  round(float(np.std(fc_vals)),  4)
    }

    agg['num_seeds'] = len(seed_results)
    return agg


# ── Main ─────────────────────────────────────────────────────────────────────

def main():
    from google.colab import drive
    drive.mount('/content/drive')

    project_root = "/content/drive/MyDrive/FinalProject"
    BASE_MODEL   = "mistralai/Mistral-7B-v0.3"

    # Load prompts
    test_file = f"{project_root}/human_baseline_data/test.jsonl"
    prompts   = load_test_prompts(test_file, n=50)
    print(f"Loaded {len(prompts)} test prompts\n")

    # Load NLI model ONCE — reused for all 4 models × 3 seeds
    nli = load_nli_model(device=0)

    models_config = {
        'Base (Untrained)': None,
        'Human-trained':    f"{project_root}/trained_models_v2/human_baseline_data_mistral",
        'AI-trained':       f"{project_root}/trained_models_v2/ai_generated_data_gpt2_medium_mistral",
        'Mixed-trained':    f"{project_root}/trained_models_v2/mixed_data_gpt2_medium_mistral"
    }

    final_results = {}

    for model_name, model_path in models_config.items():
        print("=" * 60)
        print(f"EVALUATING: {model_name}")
        print("=" * 60)

        try:
            evaluator    = ModelEvaluator(BASE_MODEL, model_path)
            seed_results = []

            for seed in SEEDS:
                print(f"\n  -- Seed {seed} --")
                set_seed(seed)
                result = run_single_seed_evaluation(evaluator, prompts, nli)
                seed_results.append(result)

            agg = aggregate_seed_results(seed_results)
            final_results[model_name] = agg

            ld = agg['lexical_diversity']
            print(f"\n{model_name} — mean ± std over {len(SEEDS)} seeds:")
            print(f"  TTR:                {ld['type_token_ratio']['mean']:.4f} ± {ld['type_token_ratio']['std']:.4f}")
            print(f"  Vocab Size:         {ld['vocabulary_size']['mean']:.1f} ± {ld['vocabulary_size']['std']:.1f}")
            print(f"  Bigram Diversity:   {ld['bigram_diversity']['mean']:.4f} ± {ld['bigram_diversity']['std']:.4f}")
            print(f"  Distinct-1:         {ld['distinct_1']['mean']:.4f} ± {ld['distinct_1']['std']:.4f}")
            print(f"  Distinct-2:         {ld['distinct_2']['mean']:.4f} ± {ld['distinct_2']['std']:.4f}")
            print(f"  Distinct-3:         {ld['distinct_3']['mean']:.4f} ± {ld['distinct_3']['std']:.4f}")
            print(f"  Self-BLEU:          {ld['self_bleu']['mean']:.4f} ± {ld['self_bleu']['std']:.4f}")
            print(f"  Perplexity:         {agg['perplexity']['mean']:.2f} ± {agg['perplexity']['std']:.2f}")
            print(f"  Factual Consistency:{agg['factual_consistency']['mean']:.4f} ± {agg['factual_consistency']['std']:.4f}")

            del evaluator
            torch.cuda.empty_cache()

        except Exception as e:
            print(f"ERROR evaluating {model_name}: {e}")
            import traceback; traceback.print_exc()
            continue

    # ── Summary table ────────────────────────────────────────────────────────
    print("\n" + "=" * 85)
    print(f"FINAL RESULTS  ({len(prompts)} prompts, {len(SEEDS)} seeds each)  —  Mistral-7B-v0.3")
    print("=" * 85)
    print(f"\n{'Model':<20} {'TTR':<18} {'Distinct-2':<18} {'Self-BLEU':<18} {'PPL':<12}")
    print("-" * 90)
    for model_name, metrics in final_results.items():
        ld  = metrics['lexical_diversity']
        ttr = f"{ld['type_token_ratio']['mean']:.4f} ± {ld['type_token_ratio']['std']:.4f}"
        d2  = f"{ld['distinct_2']['mean']:.4f} ± {ld['distinct_2']['std']:.4f}"
        sb  = f"{ld['self_bleu']['mean']:.4f} ± {ld['self_bleu']['std']:.4f}"
        ppl = f"{metrics['perplexity']['mean']:.2f} ± {metrics['perplexity']['std']:.2f}"
        print(f"{model_name:<20} {ttr:<18} {d2:<18} {sb:<18} {ppl:<12}")

    # ── Save ─────────────────────────────────────────────────────────────────
    results_path = f"{project_root}/evaluation_results_final_mistral.json"
    with open(results_path, 'w') as f:
        json.dump(final_results, f, indent=2)

    print(f"\n✓ Saved to: {results_path}")
    print("=" * 85)
    print("EVALUATION COMPLETE")
    print("=" * 85)

    return final_results


results = main()


Mounted at /content/drive
Loaded 50 test prompts

Loading NLI model (cross-encoder/nli-deberta-v3-small)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/568M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

...

Loading LoRA adapter from: /content/drive/MyDrive/FinalProject/trained_models_v2/human_baseline_data_mistral
Model loaded


  -- Seed 42 --
  Generating 50 outputs...
    10/50 done
    20/50 done
    30/50 done
    40/50 done
    50/50 done

  -- Seed 123 --
  Generating 50 outputs...
    10/50 done
    20/50 done
    30/50 done
    40/50 done
    50/50 done

  -- Seed 456 --
  Generating 50 outputs...
    10/50 done
    20/50 done
    30/50 done
    40/50 done
    50/50 done

Human-trained — mean ± std over 3 seeds:
  TTR:                0.3298 ± 0.0077
  Vocab Size:         1523.0 ± 20.0
  Bigram Diversity:   0.7721 ± 0.0078
  Distinct-1:         0.3298 ± 0.0077
  Distinct-2:         0.7721 ± 0.0078
  Distinct-3:         0.9433 ± 0.0032
  Self-BLEU:          0.0715 ± 0.0022
  Perplexity:         3.69 ± 0.14
  Factual Consistency:0.0383 ± 0.0074
EVALUATING: AI-trained
Loading model: mistralai/Mistral-7B-v0.3


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading LoRA adapter from: /content/drive/MyDrive/FinalProject/trained_models_v2/ai_generated_data_gpt2_medium_mistral
Model loaded


  -- Seed 42 --
  Generating 50 outputs...
    10/50 done
    20/50 done
    30/50 done
    40/50 done
    50/50 done

  -- Seed 123 --
  Generating 50 outputs...
    10/50 done
    20/50 done
    30/50 done
    40/50 done
    50/50 done

  -- Seed 456 --
  Generating 50 outputs...
    10/50 done
    20/50 done
    30/50 done
    40/50 done
    50/50 done

AI-trained — mean ± std over 3 seeds:
  TTR:                0.2783 ± 0.0086
  Vocab Size:         1210.7 ± 23.8
  Bigram Diversity:   0.5998 ± 0.0115
  Distinct-1:         0.2783 ± 0.0086
  Distinct-2:         0.5998 ± 0.0115
  Distinct-3:         0.7517 ± 0.0112
  Self-BLEU:          0.0620 ± 0.0061
  Perplexity:         3.40 ± 0.04
  Factual Consistency:0.0771 ± 0.0227
EVALUATING: Mixed-trained
Loading model: mistralai/Mistral-7B-v0.3


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading LoRA adapter from: /content/drive/MyDrive/FinalProject/trained_models_v2/mixed_data_gpt2_medium_mistral
Model loaded


  -- Seed 42 --
  Generating 50 outputs...
    10/50 done
    20/50 done
    30/50 done
    40/50 done
    50/50 done

  -- Seed 123 --
  Generating 50 outputs...
    10/50 done
    20/50 done
    30/50 done
    40/50 done
    50/50 done

  -- Seed 456 --
  Generating 50 outputs...
    10/50 done
    20/50 done
    30/50 done
    40/50 done
    50/50 done

Mixed-trained — mean ± std over 3 seeds:
  TTR:                0.3343 ± 0.0074
  Vocab Size:         1521.3 ± 38.6
  Bigram Diversity:   0.7762 ± 0.0106
  Distinct-1:         0.3343 ± 0.0074
  Distinct-2:         0.7762 ± 0.0106
  Distinct-3:         0.9448 ± 0.0073
  Self-BLEU:          0.0674 ± 0.0009
  Perplexity:         3.70 ± 0.06
  Factual Consistency:0.0356 ± 0.0015

FINAL RESULTS  (50 prompts, 3 seeds each)  —  Mistral-7B-v0.3

Model                TTR                Distinct-2         Self-BLEU     